In [ ]:
import json
import os
import requests
import torch
import uuid
from io import BytesIO
from PIL import Image
from pdf2image import convert_from_bytes
from colpali_engine.models import ColPali, ColPaliProcessor
from qdrant_client import QdrantClient
from qdrant_client.http import models

# ==========================================
# 1. Configuration
# ==========================================
JSON_FILE_PATH = "papers.json"         # Path to your JSON file with paper links
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION_NAME = "researchpaper_colipali"
MODEL_NAME = "vidore/colpali-v1.2"     # You can also use v1.3 or other ColPali models
VECTOR_DIM = 128                       # ColPali standard output dimension
BATCH_SIZE = 4                         # Batch size for processing images

# ==========================================
# 2. Initialize Qdrant Client & Collection
# ==========================================
qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

# Create collection with MultiVectorConfig for late interaction (ColBERT-style)
if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=VECTOR_DIM,
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            )
        )
    )
    print(f"Collection '{COLLECTION_NAME}' created.")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists.")

# ==========================================
# 3. Initialize ColPali Model
# ==========================================
print("Loading ColPali model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
# Use bfloat16 for memory efficiency if on GPU, else float32
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = ColPali.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map=device
).eval()
processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

# ==========================================
# 4. Helper Functions
# ==========================================
def download_and_convert_pdf(url):
    """Downloads a PDF from a URL and converts it to a list of PIL Images."""
    print(f"Downloading PDF from {url}...")
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    print("Converting PDF to images...")
    # Convert PDF bytes to images (one image per page)
    images = convert_from_bytes(response.content)
    return images

def get_colpali_embeddings(images):
    """Generates ColPali multivector embeddings for a list of images."""
    embeddings = []
    
    # Process in batches to avoid OOM errors
    with torch.no_grad():
        for i in range(0, len(images), BATCH_SIZE):
            batch_images = images[i:i + BATCH_SIZE]
            
            # Process inputs
            inputs = processor(images=batch_images, return_tensors="pt").to(model.device)
            
            # Forward pass
            outputs = model(**inputs)
            
            # Extract embeddings (list of 2D tensors: [num_patches, dim])
            # For Qdrant multivector, we need list of lists of floats
            batch_embeddings = outputs.tolist() 
            embeddings.extend(batch_embeddings)
            
    return embeddings

# ==========================================
# 5. Main Processing Pipeline
# ==========================================
def process_corpus():
    # Load JSON file (Assuming format: [{"url": "http://...", "title": "..."}])
    if not os.path.exists(JSON_FILE_PATH):
        print(f"Error: {JSON_FILE_PATH} not found. Please create it first.")
        return
        
    with open(JSON_FILE_PATH, 'r') as f:
        corpus = json.load(f)
        
    for item in corpus:
        pdf_url = item.get("url")
        title = item.get("title", "Unknown Title")
        
        if not pdf_url:
            continue
            
        print(f"\n--- Processing: {title} ---")
        try:
            # 1. Download & Convert
            page_images = download_and_convert_pdf(pdf_url)
            print(f"Successfully extracted {len(page_images)} pages.")
            
            # 2. Embed Pages using ColPali
            print("Generating ColPali embeddings...")
            embeddings = get_colpali_embeddings(page_images)
            
            # 3. Store in Qdrant
            print("Uploading to Qdrant...")
            points = []
            for page_num, emb in enumerate(embeddings):
                # Generate a unique ID for each page
                point_id = str(uuid.uuid4())
                
                points.append(
                    models.PointStruct(
                        id=point_id,
                        vector=emb,  # This is a MultiVector (list of vectors)
                        payload={
                            "title": title,
                            "url": pdf_url,
                            "page_number": page_num + 1,
                            "text": f"Page {page_num + 1} of {title}" # ColPali relies on image, but you can store metadata
                        }
                    )
                )
            
            qdrant_client.upsert(
                collection_name=COLLECTION_NAME,
                points=points
            )
            print(f"Successfully indexed {len(points)} pages for '{title}'.")
            
        except Exception as e:
            print(f"Failed to process {pdf_url}: {str(e)}")

if __name__ == "__main__":
    process_corpus()
    print("\n✅ Pipeline execution complete!")


Collection 'researchpaper_colipali' created.
Loading ColPali model...


adapter_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/66.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]